<a href="https://colab.research.google.com/github/shashankcm/RAG_LANGCHAIN_LLM_Projects/blob/main/gradio_chatbot_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip3 install langchain-text-splitters
!pip3 install langchain-community

!pip3 install langchain-openai langchain-core transformers huggingface_hub
!pip3 install sentence_transformers chromadb python-dotenv torch torchvision
!pip3 install langchain-chroma langchain-huggingface langchain langchain-classic
!pip3 install wget
!pip3 install gradio
!pip3 install pypdf

In [ ]:
def warn(*args, **kwargs):
  pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from google.colab import userdata

import wget
import os
import gradio as gr

api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key

# --- GLOBAL STATE TO PREVENT WIPING HISTORY & RE-EMBEDDING ---
SESSION_STORE = {}
CURRENT_VECTOR_DB = None
CURRENT_FILE_PATH = None
CONVERSATIONAL_RAG_CHAIN = None

## Document loader
def document_loader(file_path):
    loader = PyPDFLoader(file_path)
    loaded_document = loader.load()
    return loaded_document

## LLM
def get_llm():
  model = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.5,
    max_tokens=256)
  return model

## Text splitter
def text_splitter(data):
    splitter = CharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=50,
        length_function=len,
    )
    chunks = splitter.split_documents(data)
    return chunks

## Vector db
def vector_db(data):
    embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    vector_store = Chroma.from_documents(data, embedding)
    return vector_store

## Build Retriever components
def build_retriever_from_file(file_path):
  splits = document_loader(file_path)
  chunks = text_splitter(splits)
  vectordb = vector_db(chunks)
  return vectordb.as_retriever(search_kwargs={"k": 3})

## Session id - Global persistent tracking
def get_session_history(session_id: str):
    if session_id not in SESSION_STORE:
        SESSION_STORE[session_id] = InMemoryChatMessageHistory()
    return SESSION_STORE[session_id]

## QA Chain builder
def build_retriever_qa_chain(doc_retriever):
  contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question."
  )

  contextualize_q_prompt = ChatPromptTemplate.from_messages([
      ("system", contextualize_q_system_prompt),
      MessagesPlaceholder("chat_history"),
      ("human", "{input}"),
  ])

  model = get_llm()

  # Solved naming conflict by using distinct variable names
  history_aware_retriever = create_history_aware_retriever(model, doc_retriever, contextualize_q_prompt)

  qa_system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know.\n\n"
    "Context: {context}"
  )

  qa_prompt = ChatPromptTemplate.from_messages([
      ("system", qa_system_prompt),
      MessagesPlaceholder(variable_name="chat_history"),
      ("human", "{input}"),
  ])

  question_answer_chain = create_stuff_documents_chain(model, qa_prompt)
  rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

  conversational_rag_chain = RunnableWithMessageHistory(
      rag_chain,
      get_session_history,
      input_messages_key="input",
      history_messages_key="chat_history",
      output_messages_key="answer",
  )

  return conversational_rag_chain

# --- CHAT WRAPPER ---
def chatbot_qa(query, history, file):
  global CURRENT_FILE_PATH, CONVERSATIONAL_RAG_CHAIN

  if not file:
      return "Please upload a PDF document first before asking questions."

  # Check if a completely new file was uploaded or if it's the first run
  if CONVERSATIONAL_RAG_CHAIN is None or CURRENT_FILE_PATH != file:
      CURRENT_FILE_PATH = file
      doc_retriever = build_retriever_from_file(file)
      CONVERSATIONAL_RAG_CHAIN = build_retriever_qa_chain(doc_retriever)

  session_id = "gradio_user_session"
  config = {"configurable": {"session_id": session_id}}

  result = CONVERSATIONAL_RAG_CHAIN.invoke({"input": query}, config=config)
  return result['answer']

# --- MODERN GRADIO CHAT INTERFACE ---
with gr.Blocks() as demo:
    gr.Markdown("# 📄 Conversational RAG Chatbot")

    file_input = gr.File(label="Upload PDF Document", file_types=['.pdf'], type="filepath")

    # gr.ChatInterface natively manages standard chat bubble elements and historical state
    gr.ChatInterface(
        fn=chatbot_qa,
        additional_inputs=[file_input],
        title="Chat with your Data",
        description="Ask questions about your uploaded document. The assistant remembers your previous messages!"
    )

# Launch the app
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7861, share=True)
